In [1]:
!pip install -q boto3


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import threading
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import mimetypes

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv

In [3]:
env_path = Path("../.env").resolve()
load_dotenv(dotenv_path=env_path)

print("Loaded .env from:", env_path)

Loaded .env from: C:\Users\Lutfi\Documents\Project\AITF\rutilahu-vlm-etl\.env


In [4]:
def required_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        raise ValueError(f"Environment variable {name} belum diisi di .env")
    return value

# Source MinIO local
MINIO_ENDPOINT    = required_env("MINIO_ENDPOINT")
MINIO_ACCESS_KEY  = required_env("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY  = required_env("MINIO_SECRET_KEY")
MINIO_SECURE      = required_env("MINIO_SECURE").lower() in ["true", "1", "yes"]
MINIO_BUCKET_NAME = required_env("MINIO_BUCKET_NAME")

# Destination S3 / S3-compatible
S3_ENDPOINT_URL = required_env("S3_ENDPOINT_URL")
S3_ACCESS_KEY   = required_env("MINIO_USER")
S3_SECRET_KEY   = required_env("MINIO_PASSWORD")
S3_BUCKET_NAME  = required_env("S3_BUCKET_NAME")

# Local SFT dataset
SFT_LOCAL_DIR = Path("../data/sft_dataset").resolve()

MAIN_PREFIX = "MKN-2"
SFT_PREFIX  = f"{MAIN_PREFIX}/sft_dataset"

# ── Tuning parallelism ──────────────────────────────────────────────────────
# Jumlah worker thread untuk upload paralel.
# Rekomendasi: 16–32 untuk koneksi cepat, 8 jika sering timeout.
MAX_WORKERS = 16

In [5]:
# ── Logger ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

# Thread-local storage: setiap thread punya S3 client sendiri
# (boto3 client TIDAK thread-safe jika di-share antar thread)
_thread_local = threading.local()


# ── Helpers ──────────────────────────────────────────────────────────────────
def ensure_prefix(prefix: str) -> str:
    prefix = prefix.strip("/")
    return f"{prefix}/" if prefix else ""


def object_content_type(key: str) -> str:
    if key.endswith(".jsonl"):
        return "application/x-ndjson"
    guessed, _ = mimetypes.guess_type(key)
    return guessed or "application/octet-stream"


def make_s3_client(endpoint_url, access_key, secret_key, secure=True):
    """Buat boto3 S3 client baru."""
    if endpoint_url.startswith("http://") or endpoint_url.startswith("https://"):
        final_endpoint = endpoint_url
    else:
        scheme = "https" if secure else "http"
        final_endpoint = f"{scheme}://{endpoint_url}"

    return boto3.client(
        "s3",
        endpoint_url=final_endpoint,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name="us-east-1",
        config=Config(
            signature_version="s3v4",
            s3={"addressing_style": "path"},
            retries={"max_attempts": 10, "mode": "standard"},
        ),
    )


def get_thread_clients():
    """
    Ambil (atau buat) sepasang S3 client khusus untuk thread yang sedang berjalan.
    Ini memastikan tidak ada sharing state antar thread.
    """
    if not hasattr(_thread_local, "src_client"):
        _thread_local.src_client = make_s3_client(
            MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY, secure=MINIO_SECURE
        )
        _thread_local.dst_client = make_s3_client(
            S3_ENDPOINT_URL, S3_ACCESS_KEY, S3_SECRET_KEY, secure=True
        )
    return _thread_local.src_client, _thread_local.dst_client


def ensure_bucket_exists(client, bucket_name: str):
    try:
        client.head_bucket(Bucket=bucket_name)
        print(f"Bucket exists: {bucket_name}")
    except ClientError:
        try:
            client.create_bucket(Bucket=bucket_name)
            print(f"Created bucket: {bucket_name}")
        except ClientError as e:
            print(f"Bucket create skipped/failed: {e}")


def list_source_keys(client, bucket: str, prefix: str):
    """List semua object key di bawah prefix (tidak termasuk directory marker)."""
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith("/"):
                yield key


# ── Core upload function (dipanggil per thread) ───────────────────────────────
def _copy_one(src_bucket, src_key, dst_bucket, dst_key, counter, lock, total):
    """
    Copy satu object dari MinIO ke S3.
    - Selalu overwrite (tidak ada pengecekan apakah file sudah ada).
    - Menggunakan client milik thread masing-masing.
    - Print nama file yang sedang diupload + progress ke stdout.
    Return: (src_key, dst_key, None) jika sukses, (src_key, dst_key, error_str) jika gagal.
    """
    src_client, dst_client = get_thread_clients()
    filename = src_key.split("/")[-1]
    try:
        with lock:
            counter[0] += 1
            n = counter[0]
            print(f"[{n:>{len(str(total))}}/{total}] ↑ {src_key}", flush=True)

        response = src_client.get_object(Bucket=src_bucket, Key=src_key)
        body     = response["Body"]

        extra_args = {
            "ContentType": response.get("ContentType") or object_content_type(dst_key),
        }
        metadata = response.get("Metadata")
        if metadata:
            extra_args["Metadata"]          = metadata
            extra_args["MetadataDirective"] = "REPLACE"

        # upload_fileobj otomatis OVERWRITE jika key sudah ada di S3
        dst_client.upload_fileobj(
            Fileobj=body,
            Bucket=dst_bucket,
            Key=dst_key,
            ExtraArgs=extra_args,
        )
        return src_key, dst_key, None
    except Exception as exc:
        logger.error("FAILED %s -> %s: %s", src_key, dst_key, exc)
        return src_key, dst_key, str(exc)


# ── Parallel copy prefix ──────────────────────────────────────────────────────
def copy_prefix_parallel(
    src_client, src_bucket, src_prefix,
    dst_client, dst_bucket, dst_prefix,
    max_workers: int = MAX_WORKERS,
):
    """
    Copy semua object di bawah src_prefix ke dst_prefix secara paralel.
    File yang sudah ada di destination AKAN di-overwrite.
    Setiap file yang diupload di-print ke stdout dengan nomor urut.
    """
    src_prefix = ensure_prefix(src_prefix)
    dst_prefix = ensure_prefix(dst_prefix)

    print(f"\nListing objects: s3://{src_bucket}/{src_prefix} ...")
    keys = list(list_source_keys(src_client, src_bucket, src_prefix))
    total = len(keys)
    print(f"Ditemukan {total:,} objects. Mulai upload paralel ({max_workers} workers) ...\n")

    failed = []
    counter = [0]          # list agar bisa di-mutate di dalam closure
    lock    = threading.Lock()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                _copy_one,
                src_bucket,
                src_key,
                dst_bucket,
                # Pertahankan struktur path relatif di bawah prefix
                dst_prefix + (src_key[len(src_prefix):] if src_key.startswith(src_prefix) else src_key),
                counter,
                lock,
                total,
            ): src_key
            for src_key in keys
        }

        for future in as_completed(futures):
            src_key, dst_key, err = future.result()
            if err:
                failed.append((src_key, dst_key, err))

    print(f"\nSelesai: {total - len(failed):,} berhasil, {len(failed):,} gagal.")
    if failed:
        print("\n=== File yang GAGAL di-upload ===")
        for src_key, dst_key, err in failed:
            print(f"  {src_key}  ->  {dst_key}")
            print(f"    Error: {err}")
    return failed


# ── Upload local SFT dataset ──────────────────────────────────────────────────
def upload_local_file(dst_client, bucket, local_path: Path, dst_key: str):
    """Upload satu file lokal ke S3 (overwrite jika sudah ada)."""
    with local_path.open("rb") as f:
        dst_client.upload_fileobj(
            Fileobj=f,
            Bucket=bucket,
            Key=dst_key,
            ExtraArgs={"ContentType": object_content_type(str(local_path))},
        )


def upload_sft_dataset(dst_client, bucket, local_dir: Path, dst_prefix: str):
    dst_prefix      = ensure_prefix(dst_prefix)
    required_files  = ["train.jsonl", "val.jsonl", "test.jsonl"]

    for filename in required_files:
        src_path = local_dir / filename
        if not src_path.exists():
            raise FileNotFoundError(f"Tidak ditemukan: {src_path}")

        dst_key = f"{dst_prefix}{filename}"
        print(f"Upload {src_path} -> s3://{bucket}/{dst_key}")
        upload_local_file(dst_client, bucket, src_path, dst_key)
        print(f"  ✓ {filename}")

In [6]:
# ── Inisialisasi client utama (hanya untuk listing & bucket check) ────────────
src_client = make_s3_client(
    MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY, secure=MINIO_SECURE
)
dst_client = make_s3_client(
    S3_ENDPOINT_URL, S3_ACCESS_KEY, S3_SECRET_KEY, secure=True
)

ensure_bucket_exists(dst_client, S3_BUCKET_NAME)

# ── Copy image folders secara paralel ────────────────────────────────────────
# File yang sudah ada di S3 akan otomatis di-OVERWRITE.
all_failed = []
for folder_name in ["tampak_dalam", "tampak_luar", "crawled_image"]:
    failed = copy_prefix_parallel(
        src_client=src_client,
        src_bucket=MINIO_BUCKET_NAME,
        src_prefix=folder_name,
        dst_client=dst_client,
        dst_bucket=S3_BUCKET_NAME,
        dst_prefix=f"{MAIN_PREFIX}/{folder_name}",
        max_workers=MAX_WORKERS,
    )
    all_failed.extend(failed)

# ── Upload SFT dataset lokal ─────────────────────────────────────────────────
if not SFT_LOCAL_DIR.exists():
    raise FileNotFoundError(f"Folder SFT tidak ditemukan: {SFT_LOCAL_DIR}")

print("\nUpload SFT dataset ...")
upload_sft_dataset(
    dst_client=dst_client,
    bucket=S3_BUCKET_NAME,
    local_dir=SFT_LOCAL_DIR,
    dst_prefix=SFT_PREFIX,
)

# ── Ringkasan akhir ───────────────────────────────────────────────────────────
print("\n" + "="*60)
if all_failed:
    print(f"Selesai dengan {len(all_failed)} file GAGAL. Periksa log di atas.")
else:
    print("Semua proses selesai. Tidak ada file yang gagal.")
print("="*60)

Bucket exists: clean-dataset

Listing objects: s3://mkn2/tampak_dalam/ ...
Ditemukan 46,815 objects. Mulai upload paralel (16 workers) ...

[    1/46815] ↑ tampak_dalam/mkn2_interior_img_000001.jpg
[    2/46815] ↑ tampak_dalam/mkn2_interior_img_000003.jpg
[    3/46815] ↑ tampak_dalam/mkn2_interior_img_000002.jpg
[    4/46815] ↑ tampak_dalam/mkn2_interior_img_000004.jpg
[    5/46815] ↑ tampak_dalam/mkn2_interior_img_000005.jpg
[    6/46815] ↑ tampak_dalam/mkn2_interior_img_000006.jpg
[    7/46815] ↑ tampak_dalam/mkn2_interior_img_000007.jpg
[    8/46815] ↑ tampak_dalam/mkn2_interior_img_000008.jpg
[    9/46815] ↑ tampak_dalam/mkn2_interior_img_000009.jpg
[   10/46815] ↑ tampak_dalam/mkn2_interior_img_000010.jpg
[   11/46815] ↑ tampak_dalam/mkn2_interior_img_000012.jpg
[   12/46815] ↑ tampak_dalam/mkn2_interior_img_000011.jpg
[   13/46815] ↑ tampak_dalam/mkn2_interior_img_000014.jpg
[   14/46815] ↑ tampak_dalam/mkn2_interior_img_000015.jpg
[   15/46815] ↑ tampak_dalam/mkn2_interior_img_0

2026-06-17 21:45:17,089 [ERROR] FAILED crawled_image/mkn2_crawled_img_07809.jpg -> MKN-2/crawled_image/mkn2_crawled_img_07809.jpg: An error occurred while reading from response stream: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


[ 7454/10026] ↑ crawled_image/mkn2_crawled_img_07819.jpg


2026-06-17 21:45:17,055 [ERROR] FAILED crawled_image/mkn2_crawled_img_07807.jpg -> MKN-2/crawled_image/mkn2_crawled_img_07807.jpg: An error occurred while reading from response stream: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


[ 7455/10026] ↑ crawled_image/mkn2_crawled_img_07820.jpg
[ 7456/10026] ↑ crawled_image/mkn2_crawled_img_07821.jpg


...

[ 7579/10026] ↑ crawled_image/mkn2_crawled_img_07949.jpg


2026-06-17 22:17:10,613 [ERROR] FAILED crawled_image/mkn2_crawled_img_07943.jpg -> MKN-2/crawled_image/mkn2_crawled_img_07943.jpg: An error occurred while reading from response stream: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


[ 7580/10026] ↑ crawled_image/mkn2_crawled_img_07950.jpg


2026-06-17 22:17:10,623 [ERROR] FAILED crawled_image/mkn2_crawled_img_07946.jpg -> MKN-2/crawled_image/mkn2_crawled_img_07946.jpg: An error occurred while reading from response stream: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


[ 7581/10026] ↑ crawled_image/mkn2_crawled_img_07951.jpg
[ 7582/10026] ↑ crawled_image/mkn2_crawled_img_07952.jpg
[ 7583/10026] ↑ crawled_image/mkn2_crawled_img_07953.jpg
[ 7584/10026] ↑ crawled_image/mkn2_crawled_img_07954.jpg
[ 7585/10026] ↑ crawled_image/mkn2_crawled_img_07955.jpg
[ 7586/10026] ↑ crawled_image/mkn2_crawled_img_07956.jpg
[ 7587/10026] ↑ crawled_image/mkn2_crawled_img_07957.jpg
[ 7588/10026] ↑ crawled_image/mkn2_crawled_img_07958.jpg
[ 7589/10026] ↑ crawled_image/mkn2_crawled_img_07959.jpg
[ 7590/10026] ↑ crawled_image/mkn2_crawled_img_07960.jpg
[ 7591/10026] ↑ crawled_image/mkn2_crawled_img_07961.jpg
[ 7592/10026] ↑ crawled_image/mkn2_crawled_img_07962.jpg
[ 7593/10026] ↑ crawled_image/mkn2_crawled_img_07963.jpg
[ 7594/10026] ↑ crawled_image/mkn2_crawled_img_07964.jpg
[ 7595/10026] ↑ crawled_image/mkn2_crawled_img_07965.jpg
[ 7596/10026] ↑ crawled_image/mkn2_crawled_img_07966.jpg
[ 7597/10026] ↑ crawled_image/mkn2_crawled_img_07967.jpg
[ 7598/10026] ↑ crawled_image/m

In [7]:
# ── Retry manual untuk file yang gagal ───────────────────────────────────────
# Paste list failed_keys dari output error di atas.
# Jalankan cell ini tanpa perlu run ulang semua upload dari awal.

import time

FAILED_KEYS = [
    "crawled_image/mkn2_crawled_img_07809.jpg",
    "crawled_image/mkn2_crawled_img_07807.jpg",
    "crawled_image/mkn2_crawled_img_07810.jpg",
    "crawled_image/mkn2_crawled_img_07808.jpg",
    "crawled_image/mkn2_crawled_img_07815.jpg",
    "crawled_image/mkn2_crawled_img_07816.jpg",
    "crawled_image/mkn2_crawled_img_07811.jpg",
    "crawled_image/mkn2_crawled_img_07814.jpg",
    "crawled_image/mkn2_crawled_img_07805.jpg",
    "crawled_image/mkn2_crawled_img_07812.jpg",
    "crawled_image/mkn2_crawled_img_07801.jpg",
    "crawled_image/mkn2_crawled_img_07941.jpg",
    "crawled_image/mkn2_crawled_img_07945.jpg",
    "crawled_image/mkn2_crawled_img_07943.jpg",
    "crawled_image/mkn2_crawled_img_07946.jpg",
]

RETRY_MAX     = 5
RETRY_DELAY   = 3.0   # detik, berlipat tiap gagal (3s, 6s, 12s, 24s, 48s)
RETRY_WORKERS = 4     # lebih sedikit worker agar tidak overwhelm MinIO


def _retry_one(src_key):
    """Upload ulang satu file dengan exponential backoff."""
    # dst_key: ganti prefix folder asal → MAIN_PREFIX/folder
    folder = src_key.split("/")[0]
    dst_key = f"{MAIN_PREFIX}/{src_key}"

    last_exc = None
    for attempt in range(1, RETRY_MAX + 1):
        try:
            # Buat client baru tiap retry agar koneksi lama yang rusak tidak dipakai ulang
            src_c = make_s3_client(MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY, secure=MINIO_SECURE)
            dst_c = make_s3_client(S3_ENDPOINT_URL, S3_ACCESS_KEY, S3_SECRET_KEY, secure=True)

            response = src_c.get_object(Bucket=MINIO_BUCKET_NAME, Key=src_key)
            body     = response["Body"]
            extra_args = {
                "ContentType": response.get("ContentType") or object_content_type(dst_key),
            }
            metadata = response.get("Metadata")
            if metadata:
                extra_args["Metadata"]          = metadata
                extra_args["MetadataDirective"] = "REPLACE"

            dst_c.upload_fileobj(
                Fileobj=body,
                Bucket=S3_BUCKET_NAME,
                Key=dst_key,
                ExtraArgs=extra_args,
            )
            print(f"  ✓ [{attempt}/{RETRY_MAX}] {src_key}", flush=True)
            return src_key, None

        except Exception as exc:
            last_exc = exc
            wait = RETRY_DELAY * (2 ** (attempt - 1))
            print(f"  ✗ [{attempt}/{RETRY_MAX}] {src_key}: {exc}", flush=True)
            if attempt < RETRY_MAX:
                print(f"    → tunggu {wait:.0f}s lalu coba lagi...", flush=True)
                time.sleep(wait)

    return src_key, str(last_exc)


# ── Jalankan retry paralel ────────────────────────────────────────────────────
print(f"Retry {len(FAILED_KEYS)} file dengan {RETRY_WORKERS} workers ...\n")

still_failed = []
with ThreadPoolExecutor(max_workers=RETRY_WORKERS) as executor:
    futures = {executor.submit(_retry_one, k): k for k in FAILED_KEYS}
    for future in as_completed(futures):
        src_key, err = future.result()
        if err:
            still_failed.append((src_key, err))

# ── Ringkasan ─────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Berhasil : {len(FAILED_KEYS) - len(still_failed)}/{len(FAILED_KEYS)}")
print(f"Masih gagal: {len(still_failed)}")
if still_failed:
    print("\nFile yang masih gagal:")
    for src_key, err in still_failed:
        print(f"  {src_key}")
        print(f"    {err}")
print('='*60)


Retry 15 file dengan 4 workers ...

  ✓ [1/5] crawled_image/mkn2_crawled_img_07808.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07807.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07810.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07809.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07815.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07811.jpg  ✓ [1/5] crawled_image/mkn2_crawled_img_07816.jpg

  ✓ [1/5] crawled_image/mkn2_crawled_img_07814.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07812.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07805.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07801.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07941.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07945.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07943.jpg
  ✓ [1/5] crawled_image/mkn2_crawled_img_07946.jpg

Berhasil : 15/15
Masih gagal: 0
